### Introduccion

Este notebook resume la version final del proyecto de prediccion de cancelaciones hoteleras.

La decision final separa dos escenarios de negocio:

- **Modelo pre-asignacion**: se usa antes de conocer la habitacion asignada.
- **Modelo post-asignacion**: se usa cuando la habitacion ya esta asignada y puede calcularse si hubo cambio de habitacion.

### Importacion de librerias

In [ ]:
import json
import sys
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

try:
    from IPython.display import Image, display
except ImportError:
    Image = None

    def display(obj):
        print(obj)

### Definicion de constantes

In [ ]:
def buscar_raiz_proyecto(inicio: Path = Path.cwd()) -> Path:
    # Localiza la raiz del proyecto desde Jupyter o desde la carpeta del notebook.
    for candidato in [inicio, *inicio.parents]:
        if (candidato / "src").exists() and (candidato / "data").exists():
            return candidato
    raise FileNotFoundError("No se encontr? la ra?z del proyecto")


PROJECT_ROOT = buscar_raiz_proyecto()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PATH_DIRECTORIO_DATOS = PROJECT_ROOT / "data"
PATH_DIRECTORIO_DATOS_CRUDOS = PATH_DIRECTORIO_DATOS / "raw"
PATH_DIRECTORIO_DATOS_MODELADOS = PATH_DIRECTORIO_DATOS / "processed"
PATH_DIRECTORIO_OUTPUTS = PROJECT_ROOT / "outputs"
PATH_DIRECTORIO_MODELOS = PROJECT_ROOT / "models"
PATH_MODELOS_SERVING = PATH_DIRECTORIO_MODELOS / "serving"

PATH_DATASET_ORIGINAL = PATH_DIRECTORIO_DATOS_CRUDOS / "dataset_practica_final.csv"
PATH_DATASET_VICTOR = PATH_DIRECTORIO_DATOS_MODELADOS / "dataset_practica_final_modelado_victor.csv"
PATH_DATASET_ALEJANDRO = PATH_DIRECTORIO_DATOS_MODELADOS / "dataset_practica_final_modelado_alejandro.csv"
PATH_DATASET_POST_ASIGNACION = PATH_DIRECTORIO_DATOS_MODELADOS / "dataset_practica_final_modelado_post_asignacion.csv"

PATH_MODELO_PRE_ASIGNACION = PATH_MODELOS_SERVING / "pre_asignacion.joblib"
PATH_MODELO_POST_ASIGNACION = PATH_MODELOS_SERVING / "post_asignacion.joblib"

warnings.filterwarnings("ignore", message="X does not have valid feature names.*")
PROJECT_ROOT

### Lectura de datos

In [ ]:
df_raw = pd.read_csv(PATH_DATASET_ORIGINAL)
df_victor = pd.read_csv(PATH_DATASET_VICTOR)
df_alejandro = pd.read_csv(PATH_DATASET_ALEJANDRO)
df_post_asignacion = pd.read_csv(PATH_DATASET_POST_ASIGNACION)

resumen_datasets = pd.DataFrame(
    [
        {"dataset": "original", "filas": df_raw.shape[0], "columnas": df_raw.shape[1]},
        {"dataset": "victor / pre-asignaci?n", "filas": df_victor.shape[0], "columnas": df_victor.shape[1]},
        {"dataset": "alejandro", "filas": df_alejandro.shape[0], "columnas": df_alejandro.shape[1]},
        {"dataset": "post-asignaci?n", "filas": df_post_asignacion.shape[0], "columnas": df_post_asignacion.shape[1]},
    ]
)
resumen_datasets

In [ ]:
# Distribucion de la variable objetivo
balance_cancelaciones = (
    df_raw["is_canceled"]
    .value_counts(normalize=True)
    .rename(index={0: "No cancela", 1: "Cancela"})
    .mul(100)
    .round(2)
    .reset_index()
)
balance_cancelaciones.columns = ["clase", "porcentaje"]
balance_cancelaciones

### Validacion de columnas y fuga de informacion

Las columnas `reservation_status` y `reservation_status_date` se excluyen porque contienen informacion posterior al resultado de la reserva. Ademas, el proyecto distingue entre variables disponibles antes y despues de asignar habitacion.

In [ ]:
validacion_columnas = pd.DataFrame(
    [
        {
            "dataset": nombre,
            "reservation_status": "reservation_status" in df.columns,
            "reservation_status_date": "reservation_status_date" in df.columns,
            "assigned_room_type": "assigned_room_type" in df.columns,
            "room_changed": "room_changed" in df.columns,
        }
        for nombre, df in [
            ("original", df_raw),
            ("victor / pre-asignaci?n", df_victor),
            ("alejandro", df_alejandro),
            ("post-asignaci?n", df_post_asignacion),
        ]
    ]
)
validacion_columnas

### Resultados finales

In [ ]:
def leer_metricas(path: Path, escenario: str, feature_set: str) -> pd.DataFrame:
    metricas = pd.read_csv(path)
    metricas.insert(0, "escenario", escenario)
    metricas.insert(1, "feature_set", feature_set)
    return metricas


metricas_finales = pd.concat(
    [
        leer_metricas(
            PATH_DIRECTORIO_OUTPUTS / "optuna_lightgbm_docker_24h" / "metrics.csv",
            "pre-asignaci?n",
            "victor",
        ),
        leer_metricas(
            PATH_DIRECTORIO_OUTPUTS / "feature_set_post_asignacion_victor_params" / "metrics.csv",
            "post-asignaci?n",
            "post_asignacion",
        ),
        leer_metricas(
            PATH_DIRECTORIO_OUTPUTS / "optuna_post_asignacion_lgbm_8h" / "metrics.csv",
            "post-asignaci?n Optuna 8h",
            "post_asignacion",
        ),
    ],
    ignore_index=True,
)

columnas_metricas = ["escenario", "feature_set", "model", "accuracy", "precision", "recall", "f1", "roc_auc"]
metricas_finales[columnas_metricas].sort_values("f1", ascending=False)

In [ ]:
# Visualizacion comparativa de los dos modelos que quedan preparados para servir
metricas_serving = metricas_finales[
    metricas_finales["escenario"].isin(["pre-asignaci?n", "post-asignaci?n"])
].copy()

metricas_largas = metricas_serving.melt(
    id_vars=["escenario"],
    value_vars=["accuracy", "precision", "recall", "f1", "roc_auc"],
    var_name="m?trica",
    value_name="valor",
)

plt.figure(figsize=(10, 5))
sns.barplot(data=metricas_largas, x="m?trica", y="valor", hue="escenario")
plt.ylim(0.75, 1.0)
plt.title("Comparaci?n de modelos finales")
plt.ylabel("Valor")
plt.xlabel("M?trica")
plt.legend(title="Escenario")
plt.show()

### Benchmark inicial de modelos

In [ ]:
metricas_benchmark_15 = pd.read_csv(PATH_DIRECTORIO_OUTPUTS / "benchmark_15" / "metrics.csv")
metricas_benchmark_15.sort_values("f1", ascending=False).head(15)[
    ["model", "accuracy", "precision", "recall", "f1", "roc_auc"]
]

In [ ]:
metricas_finalistas = pd.read_csv(PATH_DIRECTORIO_OUTPUTS / "finalists_full" / "metrics.csv")
metricas_finalistas.sort_values("f1", ascending=False)[
    ["model", "accuracy", "precision", "recall", "f1", "roc_auc"]
]

### Artefactos graficos

In [ ]:
# Graficos generados por el pipeline final
imagenes = [
    PATH_DIRECTORIO_OUTPUTS / "optuna_lightgbm_docker_24h" / "confusion_matrix_lightgbm_optuna.png",
    PATH_DIRECTORIO_OUTPUTS / "optuna_lightgbm_docker_24h" / "roc_curve_lightgbm_optuna.png",
    PATH_DIRECTORIO_OUTPUTS / "optuna_lightgbm_docker_24h" / "feature_importance_lightgbm_optuna.png",
]

for imagen in imagenes:
    if imagen.exists():
        print(imagen.relative_to(PROJECT_ROOT))
        if Image is not None:
            display(Image(filename=str(imagen)))

### Inferencia de ejemplo

In [ ]:
from src.data import build_modeling_dataset, split_features_target

modelo_pre_asignacion = joblib.load(PATH_MODELO_PRE_ASIGNACION)
modelo_post_asignacion = joblib.load(PATH_MODELO_POST_ASIGNACION)

muestra_raw = df_raw.head(2000).copy()

predicciones = []
for nombre_modelo, feature_set, modelo in [
    ("pre_asignacion", "victor", modelo_pre_asignacion),
    ("post_asignacion", "post_asignacion", modelo_post_asignacion),
]:
    dataset_modelado = build_modeling_dataset(muestra_raw, feature_set=feature_set)
    X_muestra, y_muestra = split_features_target(dataset_modelado)
    probabilidades = modelo.predict_proba(X_muestra.head(10))[:, 1]
    clases = modelo.predict(X_muestra.head(10))

    resultado = pd.DataFrame(
        {
            "modelo": nombre_modelo,
            "real_is_canceled": y_muestra.head(10).to_numpy(),
            "pred_is_canceled": clases,
            "prob_cancelacion": probabilidades.round(4),
        }
    )
    predicciones.append(resultado)

pd.concat(predicciones, ignore_index=True)

### Como servir los modelos con FastAPI

In [ ]:
# Comandos para levantar la API desde la raiz del proyecto:
#
# $env:HOTEL_MODELS_DIR = "models/serving"
# uvicorn src.api:app --reload
#
# Swagger queda disponible en:
# http://127.0.0.1:8000/docs

### Conclusiones

El resultado final queda organizado en dos modelos LightGBM:

- El modelo **pre-asignacion** es el recomendado para actuar antes de asignar habitacion. Usa el feature set `victor` y alcanza F1 aproximado de `0.8473`.
- El modelo **post-asignacion** se usa cuando ya existe `assigned_room_type` y puede derivarse `room_changed`. Es ligeramente mas preciso, con F1 aproximado de `0.8521`.
- `reservation_status` y `reservation_status_date` se eliminan siempre porque producen fuga directa de informacion.
- La separacion de escenarios permite defender el proyecto desde negocio: no se mezcla informacion disponible antes y despues de una decision operativa.